In [6]:
from pathlib import Path
import sys
import json
from itertools import islice

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from src.metrics.evaluator import Evaluator

In [7]:
# Load goal
path = ROOT / "data" / "processed" / "GENEVA" / "test.jsonl"

with path.open("r", encoding="utf-8") as f:
    goal = [json.loads(line) for line in islice(f, 100)]

ids_goal = {
    sample["id"]: {"events": sample["events"]}
    for sample in goal
}

print(f"Loaded {len(ids_goal)} goal samples")
print(ids_goal.get("135_23605_4099597"))

Loaded 100 goal samples
None


In [8]:
num_poch = 10

poch_dir = "poch" + str(num_poch)
# Load prediction
path = ROOT / "data" / "prediction" / "GENEVA" / poch_dir / "ee_r_test.jsonl"

with path.open("r", encoding="utf-8") as f:
    prediction = [json.loads(line) for line in islice(f, 100)]

ids_prediction = {}

for sample in prediction:
    try:
        ids_prediction[sample["id"]] = json.loads(sample["result"])
    except (json.JSONDecodeError, TypeError):
        print(f"Invalid prediction JSON: {sample['id']}")

print(f"Loaded {len(ids_prediction)} valid prediction samples")

Invalid prediction JSON: 135_23605_4099606
Invalid prediction JSON: 135_23605_4099644
Invalid prediction JSON: 175_25743_4217207
Invalid prediction JSON: 195_23647_4100339
Loaded 96 valid prediction samples


In [9]:
# Lấy giao ID của hai tập
valid_ids = ids_goal.keys() & ids_prediction.keys()

ids_goal = {id_: ids_goal[id_] for id_ in valid_ids}
ids_prediction = {id_: ids_prediction[id_] for id_ in valid_ids}

print(f"Matched samples: {len(valid_ids)}")

Matched samples: 96


In [10]:
evaluator = Evaluator()

for id_ in valid_ids:
    prediction = ids_prediction[id_]
    gold = ids_goal[id_]

    evaluator.evaluate_on_sample(prediction, gold)

results = {
    "TI": evaluator.ti.compute(),
    "TC": evaluator.tc.compute(),
    "AI": evaluator.ai.compute(),
    "AC": evaluator.ac.compute(),
}

print(results)

{'TI': {'precision': 0.6358381502890174, 'recall': 0.5612244897959183, 'f1': 0.5962059620596206}, 'TC': {'precision': 0.5895953757225434, 'recall': 0.5204081632653061, 'f1': 0.5528455284552846}, 'AI': {'precision': 0.5185185185185185, 'recall': 0.4416403785488959, 'f1': 0.47700170357751276}, 'AC': {'precision': 0.5074074074074074, 'recall': 0.43217665615141954, 'f1': 0.46678023850085176}}
